# BrainDecoding — Google Drive 云盘目录一键初始化与归档核验

本 Notebook 可在任意 **Google Colab CPU 运行时** 下秒级执行，帮助你：
1. **自动创建标准云盘目录结构**（无需在网页端手动逐级新建文件夹）：
   - `/content/drive/MyDrive/BrainDecoding/archives`：上传本机打包好的 `features_meg.tar`（Pallier2025）或 `features_eeg.tar`（ChineseEEG-2）
   - `/content/drive/MyDrive/BrainDecoding/runs`：自动保存 GPU 训练产出的 `last.ckpt`、`best.ckpt`、`subjects.json` 与指标汇总
   - `/content/drive/MyDrive/BrainDecoding/data`：原始数据目录（若已直接上传 `.tar` 缓存包，此目录保持为空即可）
   - `/content/drive/MyDrive/BrainDecoding/models`：预训练模型目录（若 `.tar` 已含文本嵌入缓存，此目录保持为空即可）
2. **核验已上传的 `.tar` 缓存包完整性**：上传 `features_meg.tar` 后运行第 2 个代码单元格，即可检查文件体积与归档清单是否完整。

In [ ]:
# 1. 挂载 Google Drive 并一键创建 BrainDecoding 标准目录树
import os
import shutil
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('非 Colab 环境，跳过 drive.mount')

DRIVE_BASE_DIR = '/content/drive/MyDrive/BrainDecoding'
DRIVE_DATA_DIR = f'{DRIVE_BASE_DIR}/data'
DRIVE_ARCHIVE_DIR = f'{DRIVE_BASE_DIR}/archives'
DRIVE_RESULTS_DIR = f'{DRIVE_BASE_DIR}/runs'
DRIVE_MODELS_DIR = f'{DRIVE_BASE_DIR}/models'

LINKS_DIR = '/content/studies_links'
LOCAL_CACHE_DIR = '/content/local_cache'
VENV_DIR = '/content/proj-venv'

for folder in (
    DRIVE_BASE_DIR,
    DRIVE_DATA_DIR,
    DRIVE_ARCHIVE_DIR,
    DRIVE_RESULTS_DIR,
    DRIVE_MODELS_DIR,
    LINKS_DIR,
    LOCAL_CACHE_DIR,
):
    Path(folder).mkdir(parents=True, exist_ok=True)

os.environ['BRAINDECODING_DATA_DIR'] = LINKS_DIR
os.environ['BRAINDECODING_CACHE_DIR'] = LOCAL_CACHE_DIR
os.environ['BRAINDECODING_RESULTS_DIR'] = DRIVE_RESULTS_DIR

print('=== Google Drive 目录初始化完成 ===')
print(f'1. 缓存归档目录 (请将 features_meg.tar 上传至此): {DRIVE_ARCHIVE_DIR}')
print(f'2. 训练输出目录 (自动保存 Checkpoint 与日志)   : {DRIVE_RESULTS_DIR}')
print(f'3. 原始数据目录 (使用 .tar 缓存时可留空)       : {DRIVE_DATA_DIR}')
print(f'4. 预训练模型目录 (使用 .tar 缓存时可留空)     : {DRIVE_MODELS_DIR}')

if Path(DRIVE_BASE_DIR).exists():
    usage = shutil.disk_usage(DRIVE_BASE_DIR)
    print(f'\nGoogle Drive 剩余可用空间: {usage.free / (1024 ** 3):.2f} GiB')

In [ ]:
# 2. 核验云盘 archives/ 下的缓存归档文件是否已上传完整
import json
import tarfile
from pathlib import Path

archive_dir = Path(DRIVE_ARCHIVE_DIR)
tar_files = sorted(archive_dir.glob('*.tar'))

if not tar_files:
    print(f'当前 {archive_dir} 下尚无 .tar 文件。')
    print('请在 Google Drive 网页端进入 MyDrive -> BrainDecoding -> archives，上传本机的 archives/features_meg.tar。')
else:
    for tar_path in tar_files:
        size_gib = tar_path.stat().st_size / (1024 ** 3)
        print(f'发现归档: {tar_path.name} ({size_gib:.2f} GiB)')
        try:
            with tarfile.open(tar_path, mode='r:*') as tar:
                members = [m for m in tar.getmembers() if m.isfile()]
                print(f'  -> 归档可读，共包含 {len(members)} 个文件条目')
                if '_archive_manifest.json' in tar.getnames():
                    manifest_stream = tar.extractfile('_archive_manifest.json')
                    if manifest_stream is not None:
                        manifest = json.loads(manifest_stream.read().decode('utf-8'))
                        print(f'  -> 清单校验通过: dataset={manifest.get("dataset")}, file_count={manifest.get("file_count")}')
        except Exception as err:
            print(f'  -> [警告] 归档读取异常（可能仍在上传或同步中）: {err}')